# 09 — CICIDS2017 Combined Dataset Preparation

Loads eight cleaned CSV files, checks their feature schemas, preserves source metadata, audits class balance/duplicates/anomalies, and exports a combined CSV. **No extra rows are deleted. No training is performed.**

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Run from a VS Code notebook anywhere inside the repository.
# If the repository cannot be found, set PROJECT_ROOT manually to its folder.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next((p for p in candidates if (p / 'ml' / 'data').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find ml/data. Set PROJECT_ROOT to the repository folder.')
RAW_DIR = PROJECT_ROOT / 'ml' / 'data' / 'raw' / 'MachineLearningCVE'
PROCESSED_DIR = PROJECT_ROOT / 'ml' / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Project:', PROJECT_ROOT)
print('Raw data:', RAW_DIR)

In [ ]:
DATASET_FILES = {
    'Monday - BENIGN': 'Monday-WorkingHours-Cleaned.csv',
    'Tuesday - Brute Force': 'Tuesday-WorkingHours-Cleaned.csv',
    'Wednesday - DoS and Heartbleed': 'Wednesday-WorkingHours-Cleaned.csv',
    'Thursday Morning - Web Attacks': 'Thursday-Morning-WebAttacks-Cleaned.csv',
    'Thursday Afternoon - Infiltration': 'Thursday-Afternoon-Infiltration-Cleaned.csv',
    'Friday Morning - Botnet': 'Friday-Morning-Botnet-Cleaned.csv',
    'Friday Afternoon - PortScan': 'Friday-Afternoon-PortScan-Cleaned.csv',
    'Friday Afternoon - DDoS': 'Friday-Afternoon-DDoS-Cleaned.csv',
}
COMBINED_DIR = PROCESSED_DIR / "combined"
COMBINED_DIR.mkdir(parents=True, exist_ok=True)
for name, filename in DATASET_FILES.items():
    file = PROCESSED_DIR / filename
    print(name, "OK" if file.is_file() else "MISSING", file)
    if not file.is_file(): raise FileNotFoundError(file)

In [ ]:
LABEL_MAPPING = {
    'Botnet': 'Bot', 'BOT': 'Bot', 'bot': 'Bot',
    'Infilteration': 'Infiltration', 'INFILTRATION': 'Infiltration', 'infiltration': 'Infiltration',
    'PORTSCAN': 'PortScan', 'Portscan': 'PortScan', 'portscan': 'PortScan', 'Port Scan': 'PortScan',
    'DDos': 'DDoS', 'Ddos': 'DDoS', 'ddos': 'DDoS', 'DDOS': 'DDoS',
    'Web Attack \ufffd Brute Force': 'Web Attack - Brute Force',
    'Web Attack \ufffd XSS': 'Web Attack - XSS',
    'Web Attack \ufffd Sql Injection': 'Web Attack - SQL Injection',
    'Web Attack - Sql Injection': 'Web Attack - SQL Injection',
}
# Repair actual Unicode replacement characters in corrupted source labels.
LABEL_MAPPING = {k.replace('\ufffd', '\ufffd'): v for k, v in LABEL_MAPPING.items()}

In [ ]:
datasets = {}
for name, filename in DATASET_FILES.items():
    d = pd.read_csv(PROCESSED_DIR / filename, low_memory=False)
    d.columns = d.columns.str.strip()
    assert 'Label' in d.columns, f'{name}: missing Label'
    d['Label'] = d['Label'].astype('string').str.strip().replace(LABEL_MAPPING)
    datasets[name] = d
    print(name, d.shape, '| labels:', d['Label'].value_counts().to_dict())
reference_columns = list(next(iter(datasets.values())).columns)
assert len(reference_columns) == 79, f'Expected 79 source columns, got {len(reference_columns)}'
feature_columns = [c for c in reference_columns if c != 'Label']
assert len(feature_columns) == 78
for name, d in datasets.items():
    assert len(set(d.columns)) == len(d.columns), f'{name}: duplicate column names'
    assert set(d.columns) == set(reference_columns), f'{name}: schema mismatch'
    d = d[reference_columns].copy()
    assert all(pd.api.types.is_numeric_dtype(d[c]) for c in feature_columns), f'{name}: nonnumeric feature'
    d['SourceDataset'] = name
    d['SourceRow'] = np.arange(len(d), dtype=np.int64)
    datasets[name] = d
print('Eight schemas verified.')

In [ ]:
combined_df = pd.concat(datasets.values(), ignore_index=True)
combined_df = combined_df[feature_columns + ['SourceDataset', 'SourceRow', 'Label']]
expected_rows = sum(map(len, datasets.values()))
assert len(combined_df) == expected_rows
numeric = combined_df[feature_columns]
assert combined_df.isna().sum().sum() == 0, 'Missing values'
assert np.isinf(numeric.to_numpy(dtype=np.float64)).sum() == 0, 'Infinite values'
assert (combined_df['Flow Duration'] < 0).sum() == 0, 'Invalid duration'
print('Combined rows:', len(combined_df))
print('Combined shape:', combined_df.shape)
print('All records preserved.')

## Label counts and source overview

In [ ]:
label_counts = combined_df['Label'].value_counts()
label_summary = pd.DataFrame({'Records': label_counts, 'Percentage': (label_counts / len(combined_df) * 100).round(5)})
source_summary = pd.crosstab(combined_df['SourceDataset'], combined_df['Label'])
display(label_summary)
display(source_summary)
print('Attack records:', (combined_df['Label'] != 'BENIGN').sum())
print('Rare classes (<100):')
display(label_summary[label_summary['Records'] < 100])
label_counts.sort_values().plot.barh(figsize=(11,6), title='CICIDS2017 class distribution')
plt.tight_layout()
plt.show()

## Cross-file duplicates and potential conflicting labels

Feature signatures are hashes for a **screening audit**, not proof of identical rows. Confirm suspected conflicts by comparing raw feature values.

In [ ]:
dup_mask = combined_df.duplicated(subset=feature_columns + ['Label'])
print('Repeated identical labeled rows across merged data:', int(dup_mask.sum()))
display(combined_df.loc[dup_mask, 'Label'].value_counts().rename('Repeated').to_frame())
feature_hash = pd.util.hash_pandas_object(combined_df[feature_columns], index=False)
conflict_counts = pd.DataFrame({'hash': feature_hash.to_numpy(), 'Label': combined_df['Label'].to_numpy()}).groupby('hash')['Label'].nunique()
print('Feature hashes with multiple labels:', int((conflict_counts > 1).sum()))

## Feature audits and live feature compatibility

In [ ]:
constant = numeric.columns[numeric.nunique() <= 1].tolist()
neg_counts = (numeric < 0).sum()
negative_audit = pd.DataFrame({'Negative Count': neg_counts, 'Minimum': numeric.min()})
negative_audit = negative_audit[negative_audit['Negative Count'] > 0]
zero_audit = ((numeric == 0).mean() * 100).rename('Zero Percentage').to_frame().sort_values('Zero Percentage', ascending=False)
print('Constant features:', constant)
display(negative_audit)
display(zero_audit.head(30))
LIVE_FEATURES = ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Flow Bytes/s', 'Flow Packets/s']
assert all(c in feature_columns for c in LIVE_FEATURES)
display(numeric[LIVE_FEATURES].describe().T)
print('Verify unit consistency and flow definitions with the live TShark aggregator before training.')

## Export combined dataset and reports

In [ ]:
combined_path = COMBINED_DIR / 'CICIDS2017_Combined_Cleaned.csv'
combined_df.to_csv(combined_path, index=False)
label_summary.to_csv(COMBINED_DIR / 'CICIDS2017_Label_Summary.csv')
source_summary.to_csv(COMBINED_DIR / 'CICIDS2017_Source_Label_Matrix.csv')
negative_audit.to_csv(COMBINED_DIR / 'CICIDS2017_Negative_Feature_Audit.csv')
zero_audit.to_csv(COMBINED_DIR / 'CICIDS2017_Zero_Feature_Audit.csv')
print('Saved:', combined_path)
print('Rows:', len(combined_df), 'Features:', len(feature_columns), 'Classes:', len(label_counts))
print('Notebook 09 completed; no model has been trained.')